In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score

In [2]:
def calculate_enrichment_factor(y_true, y_prob, top_percent=0.01):
    """
    Calculates the Enrichment Factor at a given top percentage.
    EF = (Hit rate in top X%) / (Background hit rate)
    """
    # Create a dataframe of true labels and predicted probabilities
    df = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
    
    # Sort by highest predicted probability
    df = df.sort_values(by='y_prob', ascending=False)
    
    # Determine the cutoff index for the top X%
    cutoff_idx = int(len(df) * top_percent)
    
    # Fallback if the dataset is so small that 1% is 0 rows
    if cutoff_idx == 0:
        cutoff_idx = 1
        
    top_subset = df.head(cutoff_idx)
    
    actives_in_top = top_subset['y_true'].sum()
    total_actives = df['y_true'].sum()
    
    hit_rate_top = actives_in_top / cutoff_idx
    hit_rate_background = total_actives / len(df)
    
    # Handle edge case where background hit rate is 0
    if hit_rate_background == 0:
        return 0.0
        
    ef = hit_rate_top / hit_rate_background
    return ef

In [4]:
def train_and_evaluate():
    print("1. Loading ML-ready matrix...")
    data_path = "../data/processed/aromatase_ml_ready.csv"
    if not os.path.exists(data_path):
        raise FileNotFoundError("Processed data not found. Run process_data.py first.")
        
    df = pd.read_csv(data_path)
    
    # Separate features (X) and target (y)
    # The first 5 columns are metadata (ID, SMILES, standard_value, pIC50, active)
    feature_cols = [col for col in df.columns if col.startswith('fp_')]
    
    X = df[feature_cols].values
    y = df['active'].values
    
    print(f"Feature matrix shape: {X.shape}")
    
    print("2. Splitting data (80% Train, 20% Test)...")
    # stratify=y ensures the 1/0 ratio is identical in both train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("3. Training XGBoost Classifier...")
    # Initialize the model. We use specific parameters to handle the sparsity of Morgan Fingerprints.
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
    
    model.fit(X_train, y_train)
    
    print("4. Evaluating Model Metrics...")
    # We must use predict_proba for ranking and AUC, not just .predict()
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred_default = model.predict(X_test)
    
    # 1. Global Discriminative Power
    roc_auc = roc_auc_score(y_test, y_prob)
    print(f"ROC-AUC: {roc_auc:.4f} (Target > 0.85)")
    
    # 2. Virtual Screening Ranking Power
    # Check top 1% and top 5%
    ef_1 = calculate_enrichment_factor(y_test, y_prob, top_percent=0.01)
    ef_5 = calculate_enrichment_factor(y_test, y_prob, top_percent=0.05)
    
    print(f"Enrichment Factor (Top 1%):  {ef_1:.2f}x better than random")
    print(f"Enrichment Factor (Top 5%):  {ef_5:.2f}x better than random")
    
    # 3. Standard Metrics
    precision = precision_score(y_test, y_pred_default)
    recall = recall_score(y_test, y_pred_default)
    print(f"Precision (at 0.5 threshold): {precision:.4f}")
    print(f"Recall (at 0.5 threshold):    {recall:.4f}")
    
    #print("\n5. Serializing the Model Pipeline...")
    #model_dir = "../models"
    ##os.makedirs(model_dir, exist_ok=True)
    
    #model_path = os.path.join(model_dir, "aromatase_xgb_v1.joblib")
    
    # We save the model using joblib, which is heavily optimized for large numpy arrays
    #joblib.dump(model, model_path)
    #print(f"SUCCESS! Model saved to {model_path}")

In [5]:
train_and_evaluate()

1. Loading ML-ready matrix...
Feature matrix shape: (1767, 2048)
2. Splitting data (80% Train, 20% Test)...
3. Training XGBoost Classifier...
4. Evaluating Model Metrics...
ROC-AUC: 0.9391 (Target > 0.85)
Enrichment Factor (Top 1%):  1.32x better than random
Enrichment Factor (Top 5%):  1.32x better than random
Precision (at 0.5 threshold): 0.9101
Recall (at 0.5 threshold):    0.9440
